In [1]:
import os
import torch
import argparse
import sys
from pathlib import Path
from PIL import Image
from torchvision import transforms
from typing import List, Dict
from datetime import datetime

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from models import get_model


In [2]:
# Preprocessing shared by all models
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

label_names = ["Real", "Fake"]

In [11]:
def load_images(path):
    """Load one image OR all images in a folder."""
    p = Path(path)
    
    if p.is_file():
        return [p]
    elif p.is_dir():
        images = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        if len(images) == 0:
            raise ValueError("No .jpg or .png images found in folder.")
        return images
    else:
        raise ValueError(f"Invalid path: {path}")


def load_checkpoint(model, ckpt_path, device):
    """Load weights into the model."""
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model

def add_fft_channels(img_tensor):
    fft = torch.fft.fft2(img_tensor)
    real = fft.real
    return torch.cat([img_tensor, real], dim=0)   # (6, H, W)


def infer_one(model, img_tensor, device):
    """Run inference on a single tensor."""
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(out, dim=1)[0]
        pred_idx = probs.argmax().item()
    return pred_idx, probs.cpu().numpy()


def run_inference(input_path, model_ckpts, device="cuda"):
    """
    Notebook-friendly inference controller.

    Parameters:
        input_path: str — path to image or folder
        model_ckpts: dict — { model_name: [list of ckpt paths] }
        device: "cuda" or "cpu"

    Returns:
        results (list of dicts)
        results_file (txt file name)
    """

    device = torch.device(device if torch.cuda.is_available() else "cpu")

    # Load images
    image_paths = load_images(input_path)

    # Prepare output file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f"results_{timestamp}.txt"
    open(results_file, "w").close()

    def log(text):
        print(text)
        with open(results_file, "a") as f:
            f.write(text + "\n")

    log(f"Device: {device}")
    log(f"Loaded {len(image_paths)} images.")
    log("Starting inference...\n")

    all_results = []

    #   MAIN LOOP OVER MODELS AND THEIR CHECKPOINTS
    for model_name, ckpt_list in model_ckpts.items():

        if not ckpt_list:
            continue  # skip empty lists

        log(f"\n=== MODEL: {model_name} ===")

        # Build base architecture ONCE
        base_model = get_model(model_name, zero_init=False).to(device)

        for ckpt_path in ckpt_list:
            log(f"\n  Using checkpoint: {ckpt_path}")
            model = load_checkpoint(base_model, ckpt_path, device)

            # RUN MODEL ON EACH IMAGE
            for img_path in image_paths:
                img = Image.open(img_path).convert("RGB")
                img_tensor = preprocess(img)

                
                # If this is an FFT model, add FFT channels
                if "fft" in model_name.lower():
                    img_tensor = add_fft_channels(img_tensor)  # (6, H, W)

                pred_idx, probs = infer_one(model, img_tensor, device)
                pred_label = label_names[pred_idx]

                line = (f"    {img_path.name}: {pred_label} "
                        f"(Real={probs[0]:.4f}, Fake={probs[1]:.4f})")
                log(line)

                all_results.append({
                    "model": model_name,
                    "checkpoint": ckpt_path,
                    "image": img_path.name,
                    "pred_idx": pred_idx,
                    "pred_label": pred_label,
                    "prob_real": float(probs[0]),
                    "prob_fake": float(probs[1]),
                })

    log("\nInference complete.")
    return all_results, results_file


In [13]:
model_ckpts = {
    "resnet50": ["../training/best_NEW_Temporal_ResNet50_20251201_173932.pth"],
    "vit": ["../training/best_NEW_Temporal_ViT_20251201_161225.pth"],
    "resnet50_fft": ["../training/best_NEW_nonzero_Temporal_ResNet50+FFT_20251202_020614.pth"],
    "vit_fft": ["../training/best_NEW_Temporal_ViT_FFT_20251201_161700.pth"]
}

results, results_file = run_inference(
    input_path="images/",
    model_ckpts=model_ckpts,
    device="cuda"
)


Device: cuda
Loaded 3 images.
Starting inference...


=== MODEL: resnet50 ===


c:\Users\Jimmy\miniconda3\envs\id_ai\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



  Using checkpoint: ../training/best_NEW_Temporal_ResNet50_20251201_173932.pth
    fake1.jpg: Fake (Real=0.0877, Fake=0.9123)
    real1.jpg: Real (Real=0.9555, Fake=0.0445)
    real2.jpg: Real (Real=0.6633, Fake=0.3367)

=== MODEL: vit ===

  Using checkpoint: ../training/best_NEW_Temporal_ViT_20251201_161225.pth
    fake1.jpg: Fake (Real=0.0787, Fake=0.9213)
    real1.jpg: Real (Real=0.8771, Fake=0.1229)
    real2.jpg: Real (Real=0.7258, Fake=0.2742)

=== MODEL: resnet50_fft ===

  Using checkpoint: ../training/best_NEW_nonzero_Temporal_ResNet50+FFT_20251202_020614.pth
    fake1.jpg: Fake (Real=0.0000, Fake=1.0000)
    real1.jpg: Fake (Real=0.0000, Fake=1.0000)
    real2.jpg: Fake (Real=0.0000, Fake=1.0000)

=== MODEL: vit_fft ===

  Using checkpoint: ../training/best_NEW_Temporal_ViT_FFT_20251201_161700.pth
    fake1.jpg: Real (Real=0.8738, Fake=0.1262)
    real1.jpg: Real (Real=0.8284, Fake=0.1716)
    real2.jpg: Real (Real=0.8770, Fake=0.1230)

Inference complete.
